<a href="https://colab.research.google.com/github/shishirkumar12/AIMl/blob/main/RAG_PDF_Extraction_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-community faiss-cpu sentence-transformers pypdf langchain-text-splitters langchain-huggingface langchain-google-genai gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from google.colab import files
from pypdf import PdfReader
from langchain_core.documents import Document

# 1. File upload karein
uploaded = files.upload()
filename = list(uploaded.keys())[0]

Saving AWS Cloud Practitioners Exam Q&A.pdf to AWS Cloud Practitioners Exam Q&A.pdf


In [3]:
# 2. PDF Reader initialize karein
reader = PdfReader(filename)

# 3. Har page ka text uske page number metadata ke saath save karein
docs_list = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text:
        docs_list.append(Document(page_content=text, metadata={"page": i}))

print(f"Total pages loaded: {len(docs_list)}")

Total pages loaded: 299


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,   # Chunk size ko badhakar 1000 kiya
    chunk_overlap=200
)

final_chunks = text_splitter.split_documents(docs_list)
print(f"Total chunks created: {len(final_chunks)}")

Total chunks created: 531


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.from_documents(
    final_chunks,
    embeddings
)
print("Vector Database Created Successfully!")

/tmp/ipykernel_3205/891263841.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector Database Created Successfully!


In [17]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

# Fetch the key securely from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [18]:
import gradio as gr

def chat_with_pdf(question):
    # 1. Database se top 3 relevant chunks nikalein
    results = db.similarity_search(question, k=3)

    # 2. Context taiyar karein
    context = "\n\n".join([doc.page_content for doc in results])

    # 3. Prompt banayein
    prompt = f"""
    Answer only from the context.
    Context:
    {context}
    Question:
    {question}
    If the answer is not found, say:
    I couldn't find the answer in the uploaded PDF.
    """

    # 4. LLM se jawab lein
    response = llm.invoke(prompt)

    # 5. Sahi Page Numbers nikalne ka logic
    sources = set()
    for doc in results:
        page = doc.metadata.get("page")
        if isinstance(page, int):
            sources.add(f"Page {page + 1}")
        else:
            sources.add("Page Unknown")

    return response.content + "\n\nSources:\n" + "\n".join(sorted(sources))

In [ ]:
# Gradio Interface Setup
demo = gr.Interface(
    fn=chat_with_pdf,
    inputs=gr.Textbox(label="Ask Question"),
    outputs=gr.Textbox(label="Answer"),
    title="AI PDF Chatbot",
    description="Ask questions from your uploaded PDF"
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://25fb234ca7a7972ae0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
